# BITRE Road Safety Enforcement - Data Cleaning

Source: Bureau of Infrastructure and Transport Research Economics (BITRE)
Coverage: All Australian states and territories, 2008 to 2024

This notebook cleans three datasets:
- Fines, arrests and charges by offence type
- Positive breath tests and outcomes
- Positive drug tests with substance breakdown

The alcohol/drug tests conducted file (adt) is used only as a denominator for the positivity rate calculation in Section 4. It is not cleaned as a standalone dataset.

Cleaning steps applied across all datasets:
- Lowercase all column names
- Parse dates and add month columns
- Shorten long location labels
- Flag aggregate rows without removing them
- Convert Yes/No string columns to boolean
- Derive useful columns for analysis
- Export clean CSVs to data/clean/

## 1. Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 25)
pd.set_option('display.float_format', '{:,.4f}'.format)

DATA_DIR = Path('data/raw')
OUT_DIR  = Path('data/clean')
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    'fines':  DATA_DIR / 'police_enforcement_2024_fines.xlsx',
    'breath': DATA_DIR / 'police_enforcement_2024_positive_breath_tests.xlsx',
    'drugs':  DATA_DIR / 'police_enforcement_2024_positive_drug_tests.xlsx',
    'adt':    DATA_DIR / 'police_enforcement_2024_alcohol_drug_tests.xlsx',
}

LOCATION_MAP = {
    'Major Cities of Australia': 'Major Cities',
    'Inner Regional Australia':  'Inner Regional',
    'Outer Regional Australia':  'Outer Regional',
    'Remote Australia':          'Remote',
    'Very Remote Australia':     'Very Remote',
    'All regions':               'All regions',
    'Unknown':                   'Unknown',
}

print('Ready. Output folder:', OUT_DIR.resolve())

Ready. Output folder: D:\eh\Swinburne\KNIME\Police enforcement notebook\Data\clean


## 2. Helper Functions

Reusable functions shared across all three dataset sections.

In [ ]:
def load_raw(key):
    df = pd.read_excel(FILES[key])
    df.columns = df.columns.str.lower()
    print(f'[{key}] {df.shape[0]:,} rows x {df.shape[1]} cols')
    return df


def add_date_columns(df):
    for col in ('start_date', 'end_date'):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    if 'start_date' in df.columns:
        df['month']      = df['start_date'].dt.month
        df['month_name'] = df['start_date'].dt.strftime('%b')
    return df


def flag_aggregates(df):
    if 'location' in df.columns:
        df['is_all_locations']    = df['location'].eq('All regions')
        df['is_unknown_location'] = df['location'].eq('Unknown')
    if 'age_group' in df.columns:
        df['is_all_ages']         = df['age_group'].eq('All ages')
        df['is_unknown_age']      = df['age_group'].eq('Unknown')
    return df


def quality_report(df, label):
    print(f'\nQuality report: {label}')
    print(f'  Shape      : {df.shape[0]:,} rows x {df.shape[1]} cols')
    print(f'  Duplicates : {df.duplicated().sum()}')
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        print('  Nulls      : none')
    else:
        print('  Nulls:')
        for col, n in nulls.items():
            print(f'    {col}: {n:,}')


print('Helper functions ready.')

Helper functions ready.


## 3. Dataset 1 - Fines, Arrests and Charges

File: police_enforcement_2024_fines.xlsx
Shape: 12,179 rows x 11 columns

Covers four offence types: speed, mobile phone, seatbelt, and unlicensed driving.
Broken down by jurisdiction, location, age group, and detection method.
Detailed age and location breakdowns only start from 2023.

In [ ]:
fines_raw = load_raw('fines')
fines_raw.head(3)

[fines] 12,179 rows x 11 cols


,year,start_date,end_date,jurisdiction,location,age_group,metric,detection_method,fines,arrests,charges
0,2024,2024-01-01,2024-01-31,NSW,Major Cities of Australia,40-64,non_wearing_seatbelts,Police issued,151,0,2
1,2024,2024-01-01,2024-01-31,NSW,Major Cities of Australia,40-64,speed_fines,Police issued,1888,0,10
2,2024,2024-01-01,2024-01-31,NSW,Major Cities of Australia,40-64,unlicensed_driving,Not applicable,109,0,431


In [5]:
# Check unique values for categorical columns
for col in ['jurisdiction', 'location', 'age_group', 'metric', 'detection_method']:
    print(f'{col}: {sorted(fines_raw[col].unique())}')

jurisdiction: ['ACT', 'NSW', 'NT', 'QLD', 'SA', 'TAS', 'VIC', 'WA']
location: ['All regions', 'Inner Regional Australia', 'Major Cities of Australia', 'Outer Regional Australia', 'Remote Australia', 'Unknown', 'Very Remote Australia']
age_group: ['0-16', '17-25', '26-39', '40-64', '65 and over', 'All ages', 'Unknown']
metric: ['mobile_phone_use', 'non_wearing_seatbelts', 'speed_fines', 'unlicensed_driving']
detection_method: ['Average speed camera', 'Fixed camera', 'Fixed or mobile camera', 'Mobile camera', 'Not applicable', 'Other', 'Police issued', 'Red light camera', 'Unknown']


In [ ]:
print('Nulls per column:')
print(fines_raw.isnull().sum())
print('\nDuplicate rows:', fines_raw.duplicated().sum())
print('Negative outcome values:', (fines_raw[['fines', 'arrests', 'charges']] < 0).any().any())

Nulls per column:
year                0
start_date          0
end_date            0
jurisdiction        0
location            0
age_group           0
metric              0
detection_method    0
fines               0
arrests             0
charges             0
dtype: int64

Duplicate rows: 0
Negative outcome values: False


In [ ]:
fines = fines_raw.copy()

fines = add_date_columns(fines)

fines['location'] = fines['location'].map(LOCATION_MAP)

fines = flag_aggregates(fines)

# Simplify detection method to short labels
fines['detection_method'] = fines['detection_method'].map({
    'Police issued':  'police',
    'Camera':         'camera',
    'Not applicable': np.nan,
})

# Add short display label for each offence type
fines['metric_label'] = fines['metric'].map({
    'speed_fines':           'Speed',
    'mobile_phone_use':      'Mobile Phone',
    'non_wearing_seatbelts': 'Seatbelt',
    'unlicensed_driving':    'Unlicensed Driving',
})

fines['pre_2023'] = fines['year'] < 2023

fines['total_actions'] = fines['fines'] + fines['arrests'] + fines['charges']

quality_report(fines, 'fines')


Quality report: fines
  Shape      : 12,179 rows x 20 cols
  Duplicates : 644
  Nulls:
    detection_method: 6,415


In [ ]:
(
    fines[fines['is_all_locations'] & fines['is_all_ages']]
    .groupby('metric_label')[['fines', 'arrests', 'charges']]
    .sum()
    .sort_values('fines', ascending=False)
)

,fines,arrests,charges
metric_label,,,
Speed,52094900,0,0
Mobile Phone,2450159,0,0
Seatbelt,290139,0,0


In [ ]:
fines.to_csv(OUT_DIR / 'fines_clean.csv', index=False)
print('Saved: data/clean/fines_clean.csv')
print(f'Shape: {fines.shape[0]:,} rows x {fines.shape[1]} cols')
print('Columns:', list(fines.columns))

Saved: data/clean/fines_clean.csv
Shape: 12,179 rows x 20 cols
Columns: ['year', 'start_date', 'end_date', 'jurisdiction', 'location', 'age_group', 'metric', 'detection_method', 'fines', 'arrests', 'charges', 'month', 'month_name', 'is_all_locations', 'is_unknown_location', 'is_all_ages', 'is_unknown_age', 'metric_label', 'pre_2023', 'total_actions']


## 4. Dataset 2 - Positive Breath Tests

File: police_enforcement_2024_positive_breath_tests.xlsx
Shape: 1,326 rows x 12 columns

Records the number of positive breath tests and enforcement outcomes (fines, arrests, charges)
broken down by jurisdiction, location, and age group.

We also merge this with the adt file to compute the drink-drive positivity rate,
which is the share of all tests that returned a positive result.

In [10]:
# Load and preview
breath_raw = load_raw('breath')
breath_raw.head(3)

[breath] 1,326 rows x 12 cols


,year,start_date,end_date,jurisdiction,location,age_group,metric,detection_method,count,fines,arrests,charges
0,2008,2008-01-01,2008-12-31,ACT,All regions,All ages,positive_breath_tests,Not applicable,1887,0,0,0
1,2008,2008-01-01,2008-12-31,NSW,All regions,All ages,positive_breath_tests,Not applicable,27368,0,0,0
2,2008,2008-01-01,2008-12-31,NT,All regions,All ages,positive_breath_tests,Not applicable,9460,0,0,0


In [ ]:
for col in ['jurisdiction', 'location', 'age_group', 'metric', 'detection_method']:
    print(f'{col}: {sorted(breath_raw[col].unique())}')

jurisdiction: ['ACT', 'NSW', 'NT', 'QLD', 'SA', 'TAS', 'VIC', 'WA']
location: ['All regions', 'Inner Regional Australia', 'Major Cities of Australia', 'Outer Regional Australia', 'Remote Australia', 'Unknown', 'Very Remote Australia']
age_group: ['0-16', '17-25', '26-39', '40-64', '65 and over', 'All ages', 'Unknown']
metric: ['positive_breath_tests']
detection_method: ['Not applicable']


In [ ]:
print('Nulls per column:')
print(breath_raw.isnull().sum())
print('\nDuplicate rows:', breath_raw.duplicated().sum())
print('Zero-count rows:', (breath_raw['count'] == 0).sum())

Nulls per column:
year                0
start_date          0
end_date            0
jurisdiction        0
location            0
age_group           0
metric              0
detection_method    0
count               0
fines               0
arrests             0
charges             0
dtype: int64

Duplicate rows: 0
Zero-count rows: 120


In [ ]:
breath = breath_raw.copy()

breath = add_date_columns(breath)

breath['location'] = breath['location'].map(LOCATION_MAP)

breath = flag_aggregates(breath)

breath = breath.rename(columns={'count': 'positive_breath_tests'})

assert breath['metric'].nunique() == 1
breath = breath.drop(columns=['metric'])

assert breath['detection_method'].nunique() == 1
breath = breath.drop(columns=['detection_method'])

# Total outcomes per row and rate of outcomes per positive test
# outcome_rate shows how often a positive test leads to a fine, arrest, or charge
breath['total_outcomes'] = breath['fines'] + breath['arrests'] + breath['charges']
breath['outcome_rate'] = np.where(
    breath['positive_breath_tests'] > 0,
    (breath['total_outcomes'] / breath['positive_breath_tests']).round(4),
    np.nan
)


breath['pre_2023'] = breath['year'] < 2023

quality_report(breath, 'breath')


Quality report: breath
  Shape      : 1,326 rows x 19 cols
  Duplicates : 0
  Nulls:
    outcome_rate: 120


In [ ]:
# Formula: positive_breath_tests / breath_tests_conducted * 100

adt_raw = load_raw('adt')
adt_breath = (
    adt_raw[adt_raw['metric'] == 'breath_tests_conducted']
    .groupby(['year', 'jurisdiction'], as_index=False)['count']
    .sum()
    .rename(columns={'count': 'breath_tests_total'})
)

# Aggregate positive tests to one row per year and jurisdiction
breath_agg = (
    breath[breath['is_all_locations'] & breath['is_all_ages']]
    .groupby(['year', 'jurisdiction'], as_index=False)['positive_breath_tests']
    .sum()
)

# Merge and calculate the rate
positivity = breath_agg.merge(adt_breath, on=['year', 'jurisdiction'], how='left')
positivity['positivity_rate_pct'] = (
    positivity['positive_breath_tests'] / positivity['breath_tests_total'] * 100
).round(4)

print('Top 10 highest positivity rates:')
positivity.sort_values('positivity_rate_pct', ascending=False).head(10)

[adt] 530 rows x 7 cols
Top 10 highest positivity rates:


,year,jurisdiction,positive_breath_tests,breath_tests_total,positivity_rate_pct
2,2008,NT,9460,111662,8.4720
42,2013,NT,12395,156719,7.9091
58,2015,NT,14192,181843,7.8045
112,2022,NT,5120,65819,7.7789
50,2014,NT,13562,175914,7.7094
98,2020,NT,7362,103102,7.1405
10,2009,NT,10990,159374,6.8957
105,2021,NT,5492,81987,6.6986
18,2010,NT,10842,181626,5.9694
34,2012,NT,9332,167667,5.5658


In [ ]:
breath.to_csv(OUT_DIR / 'breath_clean.csv', index=False)
positivity.to_csv(OUT_DIR / 'breath_positivity_rate.csv', index=False)

print('Saved: data/clean/breath_clean.csv')
print(f'Shape: {breath.shape[0]:,} rows x {breath.shape[1]} cols')
print('Columns:', list(breath.columns))
print('\nSaved: data/clean/breath_positivity_rate.csv')
print(f'Shape: {positivity.shape[0]:,} rows x {positivity.shape[1]} cols')

Saved: data/clean/breath_clean.csv
Shape: 1,326 rows x 19 cols
Columns: ['year', 'start_date', 'end_date', 'jurisdiction', 'location', 'age_group', 'positive_breath_tests', 'fines', 'arrests', 'charges', 'month', 'month_name', 'is_all_locations', 'is_unknown_location', 'is_all_ages', 'is_unknown_age', 'total_outcomes', 'outcome_rate', 'pre_2023']

Saved: data/clean/breath_positivity_rate.csv
Shape: 117 rows x 5 cols


## 5. Dataset 3 - Positive Drug Tests

File: police_enforcement_2024_positive_drug_tests.xlsx
Shape: 7,982 rows x 21 columns

Records positive drug tests with per-substance flag columns: amphetamine, cannabis, cocaine,
ecstasy, methylamphetamine, other, unknown, and no_drugs_detected.
These are stored as Yes/No strings and will be converted to booleans.

A single test can flag more than one substance at the same time.

In [ ]:
drugs_raw = load_raw('drugs')
drugs_raw.head(3)

[drugs] 7,982 rows x 21 cols


,year,start_date,end_date,jurisdiction,location,age_group,metric,best_detection_method,detection_method,amphetamine,cannabis,cocaine,ecstasy,methylamphetamine,other,unknown,no_drugs_detected,count,fines,arrests,charges
0,2008,2008-01-01,2008-12-31,NSW,All regions,All ages,positive_drug_tests,Yes,Indicator (Stage 1),No,No,No,No,No,No,Yes,No,542,0,0,0
1,2008,2008-01-01,2008-12-31,QLD,All regions,All ages,positive_drug_tests,Yes,Indicator (Stage 1),No,No,No,No,No,No,Yes,No,216,0,0,0
2,2008,2008-01-01,2008-12-31,SA,All regions,All ages,positive_drug_tests,Yes,Indicator (Stage 1),No,No,No,No,No,No,Yes,No,600,0,0,0


In [ ]:
for col in ['jurisdiction', 'location', 'age_group', 'detection_method', 'best_detection_method']:
    print(f'{col}: {sorted(drugs_raw[col].unique())}')

DRUG_FLAG_COLS = [
    'amphetamine', 'cannabis', 'cocaine', 'ecstasy',
    'methylamphetamine', 'other', 'unknown', 'no_drugs_detected'
]

print('\nDrug flag unique values:')
for col in DRUG_FLAG_COLS:
    print(f'  {col}: {drugs_raw[col].unique().tolist()}')

jurisdiction: ['ACT', 'NSW', 'NT', 'QLD', 'SA', 'TAS', 'VIC', 'WA']
location: ['All regions', 'Inner Regional Australia', 'Major Cities of Australia', 'Outer Regional Australia', 'Remote Australia', 'Unknown', 'Very Remote Australia']
age_group: ['0-16', '17-25', '26-39', '40-64', '65 and over', 'All ages', 'Unknown']
detection_method: ['Indicator (Stage 1)', 'Laboratory or Toxicology (Stage 3)', 'Not applicable', 'Secondary Confirmatory (Stage 2)']
best_detection_method: ['No', 'Yes']

Drug flag unique values:
  amphetamine: ['No', 'Yes', 'Not applicable']
  cannabis: ['No', 'Yes', 'Not applicable']
  cocaine: ['No', 'Yes', 'Not applicable']
  ecstasy: ['No', 'Not applicable', 'Yes']
  methylamphetamine: ['No', 'Not applicable', 'Yes']
  other: ['No', 'Yes', 'Not applicable']
  unknown: ['Yes', 'No', 'Not applicable']
  no_drugs_detected: ['No', 'Not applicable', 'Yes']


In [ ]:
print('Nulls per column:')
print(drugs_raw.isnull().sum())
print('\nDuplicate rows:', drugs_raw.duplicated().sum())
print('Zero-count rows:', (drugs_raw['count'] == 0).sum())

Nulls per column:
year                     0
start_date               0
end_date                 0
jurisdiction             0
location                 0
age_group                0
metric                   0
best_detection_method    0
detection_method         0
amphetamine              0
cannabis                 0
cocaine                  0
ecstasy                  0
methylamphetamine        0
other                    0
unknown                  0
no_drugs_detected        0
count                    0
fines                    0
arrests                  0
charges                  0
dtype: int64

Duplicate rows: 0
Zero-count rows: 490


In [ ]:
drugs = drugs_raw.copy()

drugs = add_date_columns(drugs)

drugs['location'] = drugs['location'].map(LOCATION_MAP)

drugs = flag_aggregates(drugs)

drugs = drugs.rename(columns={'count': 'positive_drug_tests'})

assert drugs['metric'].nunique() == 1
drugs = drugs.drop(columns=['metric'])

YES_NO = {'Yes': True, 'No': False}
for col in DRUG_FLAG_COLS:
    drugs[col] = drugs[col].map(YES_NO)

drugs['best_detection_method'] = drugs['best_detection_method'].map(YES_NO)

SUBSTANCE_COLS   = ['amphetamine', 'cannabis', 'cocaine', 'ecstasy', 'methylamphetamine', 'other', 'unknown']
SUBSTANCE_LABELS = ['Amphetamine', 'Cannabis', 'Cocaine', 'Ecstasy', 'Methylamphetamine', 'Other', 'Unknown']

def make_combo(row):
    if row.get('no_drugs_detected', False):
        return 'None detected'
    flagged = [lbl for col, lbl in zip(SUBSTANCE_COLS, SUBSTANCE_LABELS) if row.get(col, False)]
    return ' + '.join(flagged) if flagged else 'Unknown'

drugs['substance_combo'] = drugs.apply(make_combo, axis=1)

# Mark rows before 2023
drugs['pre_2023'] = drugs['year'] < 2023

drugs['total_outcomes'] = drugs['fines'] + drugs['arrests'] + drugs['charges']

quality_report(drugs, 'drugs')


Quality report: drugs
  Shape      : 7,982 rows x 29 cols
  Duplicates : 0
  Nulls:
    amphetamine: 1,027
    cannabis: 285
    cocaine: 2,341
    ecstasy: 3,315
    methylamphetamine: 7,170
    other: 7,657
    unknown: 6,276
    no_drugs_detected: 6,780


In [ ]:
drugs[[
    'year', 'jurisdiction', 'age_group', 'location',
    'detection_method', 'positive_drug_tests',
    'amphetamine', 'cannabis', 'methylamphetamine',
    'substance_combo', 'total_outcomes'
]].head(6)

,year,jurisdiction,age_group,location,detection_method,positive_drug_tests,amphetamine,cannabis,methylamphetamine,substance_combo,total_outcomes
0,2008,NSW,All ages,All regions,Indicator (Stage 1),542,False,False,False,Unknown,0
1,2008,QLD,All ages,All regions,Indicator (Stage 1),216,False,False,False,Unknown,0
2,2008,SA,All ages,All regions,Indicator (Stage 1),600,False,False,False,Unknown,0
3,2008,TAS,All ages,All regions,Indicator (Stage 1),211,False,False,False,Unknown,0
4,2008,VIC,All ages,All regions,Indicator (Stage 1),438,False,False,False,Unknown,0
5,2008,WA,All ages,All regions,Indicator (Stage 1),406,False,False,False,Unknown,0


In [ ]:
# Build a year-level substance summary table
# Summing boolean columns counts how many tests flagged each substance
drug_summary = (
    drugs[drugs['is_all_locations'] & drugs['is_all_ages']]
    .groupby('year')[SUBSTANCE_COLS + ['positive_drug_tests']]
    .sum()
    .reset_index()
)

for col in SUBSTANCE_COLS:
    drug_summary[f'{col}_pct'] = (
        drug_summary[col] / drug_summary['positive_drug_tests'] * 100
    ).round(2)

print('Substance detection counts by year:')
drug_summary[['year'] + SUBSTANCE_COLS + ['positive_drug_tests']]

Substance detection counts by year:


,year,amphetamine,cannabis,cocaine,ecstasy,methylamphetamine,other,unknown,positive_drug_tests
0,2008,0,0,0,0,0,0,6,2413
1,2009,0,0,0,0,0,0,6,2910
2,2010,0,0,0,0,0,0,5,4033
3,2011,0,0,0,0,0,0,6,5604
4,2012,0,0,0,0,0,0,7,8242
5,2013,0,0,0,0,0,0,8,9853
6,2014,0,0,0,0,0,0,8,16289
7,2015,0,0,0,0,0,0,8,35143
8,2016,0,0,0,0,0,0,8,38703
9,2017,0,0,0,0,0,0,8,39855


In [ ]:
drugs.to_csv(OUT_DIR / 'drugs_clean.csv', index=False)
drug_summary.to_csv(OUT_DIR / 'drugs_substance_by_year.csv', index=False)

print('Saved: data/clean/drugs_clean.csv')
print(f'Shape: {drugs.shape[0]:,} rows x {drugs.shape[1]} cols')
print('Columns:', list(drugs.columns))
print('\nSaved: data/clean/drugs_substance_by_year.csv')
print(f'Shape: {drug_summary.shape[0]:,} rows x {drug_summary.shape[1]} cols')

Saved: data/clean/drugs_clean.csv
Shape: 7,982 rows x 29 cols
Columns: ['year', 'start_date', 'end_date', 'jurisdiction', 'location', 'age_group', 'best_detection_method', 'detection_method', 'amphetamine', 'cannabis', 'cocaine', 'ecstasy', 'methylamphetamine', 'other', 'unknown', 'no_drugs_detected', 'positive_drug_tests', 'fines', 'arrests', 'charges', 'month', 'month_name', 'is_all_locations', 'is_unknown_location', 'is_all_ages', 'is_unknown_age', 'substance_combo', 'pre_2023', 'total_outcomes']

Saved: data/clean/drugs_substance_by_year.csv
Shape: 15 rows x 16 cols


## 6. Final Summary

In [ ]:
cleaned_datasets = {
    'fines_clean.csv':             fines,
    'breath_clean.csv':            breath,
    'breath_positivity_rate.csv':  positivity,
    'drugs_clean.csv':             drugs,
    'drugs_substance_by_year.csv': drug_summary,
}

print(f'{"File":<35} {"Rows":>8} {"Cols":>6} {"Nulls":>8} {"Dupes":>8}')
print('-' * 68)
for name, df in cleaned_datasets.items():
    nulls = df.isnull().sum().sum()
    dupes = df.duplicated().sum()
    print(f'{name:<35} {len(df):>8,} {df.shape[1]:>6} {nulls:>8,} {dupes:>8}')

File                                    Rows   Cols    Nulls    Dupes
--------------------------------------------------------------------
fines_clean.csv                       12,179     20    6,415      644
breath_clean.csv                       1,326     19      120        0
breath_positivity_rate.csv               117      5        0        0
drugs_clean.csv                        7,982     29   34,851        0
drugs_substance_by_year.csv               15     16        0        0


In [ ]:
notes = """
Some nulls are expected and intentional:

fines_clean.csv
  detection_method is NaN where the original value was 'Not applicable'
  This applies to unlicensed driving which cannot be detected by camera

breath_clean.csv
  outcome_rate is NaN where positive_breath_tests equals zero
  This prevents division by zero

breath_positivity_rate.csv
  breath_tests_total and positivity_rate_pct are NaN for jurisdictions
  and years that have no matching record in the adt file

All other columns across all files have zero nulls.
"""
print(notes)


Some nulls are expected and intentional:

fines_clean.csv
  detection_method is NaN where the original value was 'Not applicable'
  This applies to unlicensed driving which cannot be detected by camera

breath_clean.csv
  outcome_rate is NaN where positive_breath_tests equals zero
  This prevents division by zero

breath_positivity_rate.csv
  breath_tests_total and positivity_rate_pct are NaN for jurisdictions
  and years that have no matching record in the adt file

All other columns across all files have zero nulls.

